# Kalman Class

* A private function for finding hyperparameters
* A private function for the forward step in a Kalman filter
* A private function for the smoother of a Kalman filter
* A public funciton to return the results of the Kalman filter

In [1]:
import numpy as np

In [ ]:
class Kalman:
    def __init__(self, X, delta_t):
        self.X = X
        self.delta_t = delta_t

        self._F = np.identity(2)
        self._H = np.array([[1, 0]])
        self._Q = np.identity(2)
        self._P = np.identity(2)
        self._R = np.array([[1]])

    def _forward_step(self,
                      z: int, 
                      t: int,
                      X_prev: np.array, 
                      P_prev: np.array):
        self._F[0][1] = t;

        X_pred = self._F @ X_prev

        P_pred = self._F @ P_prev @ self._F.T + self._Q * t

        K_gain = P_pred @ self._H.T @ np.linalg.inv(self._H @ P_pred @ self._H.T + self._R)

        X = X_pred + K_gain @ (z - self._H @ X_pred)

        P = (np.identity(2) - K_gain @ self._H) @ P_pred

        return X, X_pred, P, P_pred

    def _forward(self):
        var_r = np.var(self.X, ddof=1)
        var_r_dot = np.var(np.diff(self.X), ddof=1)

        X_arr = [np.array([[0], 
                           [0]])]
        X_pred_arr = [np.array([[0],
                                [0]])]
        P_arr = [np.array([[var_r, 0], 
                               [0, var_r_dot]])]
        P_pred_arr = [np.array([[var_r, 0],
                               [0, var_r_dot]])]
        

        for i in range(len(self.X)):
            X, X_pred, P, P_pred = self._forward_step(z = self.X[i],
                                      t = self.delta_t[i],
                                      X_prev = X_arr[i],
                                      P_prev = P_arr[i])
            P_arr.append(P)
            X_arr.append(X)
            X_pred_arr.append(X_pred)
            P_pred_arr.append(P_pred)

        X_arr.pop(0)
        X_pred_arr.pop(0)
        P_arr.pop(0)
        P_pred_arr.pop(0)

        return X_arr, X_pred_arr, P_arr, P_pred_arr

    def _smoother_step(self, 
                      X, X_pred_next,
                      X_smooth_next,
                      P, P_pred_next,
                      P_smooth_next,
                      t):
        self._F[0][1] = t

        G_gain = P @ self._F.T @ np.linalg.inv(P_pred_next)

        smooth_X = X + G_gain @ (X_smooth_next - X_pred_next)
        smooth_P = P + G_gain @ (P_smooth_next - P_pred_next) @ G_gain.T

        return smooth_X, smooth_P, G_gain

def _smoother(self):
    X_arr, X_pred_arr, P_arr, P_pred_arr = self._forward()

    smooth_X_arr = [X_arr[-1]]
    smooth_P_arr = [P_arr[-1]]
    smooth_cross_cov = []

    for i in range(len(self.X) - 2, -1, -1):
        smooth_X, smooth_P, G_gain = self._smoother_step(
            X_arr[i],     X_pred_arr[i + 1],
            smooth_X_arr[-1],
            P_arr[i],     P_pred_arr[i + 1],
            smooth_P_arr[-1],
            self.delta_t[i]
        )
        
        smooth_cross_cov.append(G_gain @ smooth_P_arr[-1])
        smooth_X_arr.append(smooth_X)
        smooth_P_arr.append(smooth_P)

    return (
        smooth_X_arr[::-1],
        smooth_P_arr[::-1],
        smooth_cross_cov[::-1]
    )

def _log_likelihood(self):
    _, X_pred_arr, _, P_pred_arr = self._forward()
    T = len(self.X)
    ll = 0.0
    for i in range(T):
        S = (self._H @ P_pred_arr[i] @ self._H.T + self._R)[0, 0]
        innov = self.X[i] - (self._H @ X_pred_arr[i])[0, 0]
        ll += -0.5 * (np.log(S) + innov ** 2 / S)
    return ll

def _EM_step(self):
    X_arr, P_arr, cross_cov = self._smoother()
    T = len(self.X)

    R_new = np.zeros((1, 1))
    for i in range(T):
        residual = self.X[i] - (self._H @ X_arr[i])[0, 0]
        R_new[0, 0] += residual ** 2 + (self._H @ P_arr[i] @ self._H.T)[0, 0]
    self._R = R_new / T

    Q_new = np.zeros((2, 2))
    for i in range(1, T):
        xt = X_arr[i]
        xt_prev = X_arr[i - 1]
        Pt = P_arr[i]
        Pt_prev = P_arr[i - 1]
        Ct = cross_cov[i] + xt @ xt_prev.T

        Q_new += (
            Pt + xt @ xt.T
            - self._F @ Ct.T
            - Ct @ self._F.T
            + self._F @ (Pt_prev + xt_prev @ xt_prev.T) @ self._F.T
        )
    self._Q = Q_new / T

def EM(self, tol=1e-6, max_iter=100):
    ll_prev = self._log_likelihood()

    for i in range(max_iter):
        self._EM_step()
        ll = self._log_likelihood()

        if abs(ll - ll_prev) < tol:
            print(f"EM converged at iteration {i + 1}")
            break

        ll_prev = ll
    else:
        print(f"EM did not converge within {max_iter} iterations")